# 08.P · ShopWave product image classifier

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/08-deep-learning/project/project.ipynb)

Read the [brief](README.md) first. Fill in every `TODO` cell; the headings follow the rubric. The data loading is done for you, so you can spend your time on the modelling and the analysis.

A GPU (or Colab) makes training much faster; on a laptop CPU allow 10–20 minutes for the main training run.

In [ ]:
import os, warnings
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"        # hide TensorFlow's C++ start-up messages
warnings.filterwarnings("ignore")                # hide library deprecation notices
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import layers
keras.utils.set_random_seed(42)                  # same start every run (results still vary slightly by machine)
print("Keras", keras.__version__, "· backend", keras.backend.backend())

import pathlib

def flower_folder():
    """Download the 5-class flower photos once (229 MB) and return the folder."""
    path = keras.utils.get_file(
        "flower_photos", untar=True,
        origin="https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz")
    return next(p.parent for p in pathlib.Path(path).rglob("daisy") if p.is_dir())

def load_flowers(n_train=200, n_val=40, n_test=100, size=160, seed=0):
    """Pick n images per class for train/val/test; return uint8 arrays (small in memory)."""
    root = flower_folder()
    classes = sorted(p.name for p in root.iterdir() if p.is_dir())
    rng = np.random.default_rng(seed)
    parts = {"train": (0, n_train), "val": (n_train, n_train + n_val),
             "test": (n_train + n_val, n_train + n_val + n_test)}
    data = {k: ([], []) for k in parts}
    for label, name in enumerate(classes):
        files = sorted((root / name).glob("*.jpg"))
        rng.shuffle(files)
        for k, (a, b) in parts.items():
            for f in files[a:b]:
                img = keras.utils.load_img(f, target_size=(size, size))
                data[k][0].append(np.asarray(img, dtype="uint8"))
                data[k][1].append(label)
    out = {k: (np.stack(v[0]), np.array(v[1])) for k, v in data.items()}
    return out, classes

from pathlib import Path

def models_dir():
    """The repo-root models/ folder (git-ignored); created if missing."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / "data").is_dir() and (parent / "modules").is_dir():
            folder = parent / "models"
            break
    else:
        folder = Path("models")
    folder.mkdir(exist_ok=True)
    return folder

def load_all_flowers(size=160, val_share=0.15, test_share=0.15, seed=0):
    """Every photo, split per class into train / validation / test. uint8 keeps memory small."""
    root = flower_folder()
    classes = sorted(p.name for p in root.iterdir() if p.is_dir())
    rng = np.random.default_rng(seed)
    split = {"train": ([], []), "val": ([], []), "test": ([], [])}
    for label, name in enumerate(classes):
        files = sorted((root / name).glob("*.jpg"))
        rng.shuffle(files)
        n_val, n_test = round(len(files) * val_share), round(len(files) * test_share)
        parts = {"val": files[:n_val], "test": files[n_val:n_val + n_test], "train": files[n_val + n_test:]}
        for k, fs in parts.items():
            for f in fs:
                split[k][0].append(np.asarray(keras.utils.load_img(f, target_size=(size, size)), dtype="uint8"))
                split[k][1].append(label)
    return {k: (np.stack(v[0]), np.array(v[1])) for k, v in split.items()}, classes

data, CLASSES = load_all_flowers()
(x_train, y_train), (x_val, y_val), (x_test, y_test) = data["train"], data["val"], data["test"]
print("classes:", CLASSES)
print("train", len(y_train), " val", len(y_val), " test", len(y_test))

## 1. Look at the data

TODO: show a few photos per class, the number of photos per class in each split, and the accuracy of always predicting the largest class.

In [ ]:
# TODO

## 2. Build the model

TODO: a pretrained base (frozen), data augmentation, dropout and a new output layer. Print the number of trainable and frozen weights.

In [ ]:
# TODO

## 3. Train with early stopping

TODO: train with `keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)` for at most 15 epochs. Plot training and validation accuracy.

In [ ]:
# TODO

## 4. Evaluate on the test set

TODO: test accuracy, a per-class report (`sklearn.metrics.classification_report`) and a confusion matrix.

In [ ]:
# TODO

## 5. Error analysis

TODO: show at least 10 misclassified test photos with true label, predicted label and confidence. Group what you see (wrong label? two flowers in one photo? unusual angle?).

In [ ]:
# TODO

## 6. A confidence rule for the tagging team

TODO: for confidence thresholds 0.5–0.95, compute the share of photos the model would tag on its own and its accuracy on those. Pick a threshold and justify it.

In [ ]:
# TODO

## 7. Save, reload, and check

TODO: save the model to the repo's `models/` folder with `model.save(...)`, load it back with `keras.models.load_model`, and confirm it gives the same predictions.

In [ ]:
# TODO

## 8. Ready to deploy?

TODO: in a short paragraph for the head of e-commerce: how good is it, where does it fail, what should happen to low-confidence photos, and what would you check before going live?